# KECERDASAN ARTIFISIAL UNTUK GEOFISIKA
## Tugas 2 AI: Prompt dan Penyelesaian Masalah Geofisika
**Nama:** Indira Nazwa Nandita  
**NIM:** H061231054  
**Kelas:** B

Notebook ini mengikuti susunan lima kasus pada notebook referensi Tugas 2 Rahmat Hidayat. Data yang disediakan adalah **data simulasi untuk latihan**, bukan data pengukuran lapangan. Jalankan sel dari atas ke bawah. Letakkan semua file CSV di folder yang sama dengan notebook atau sesuaikan variabel `DATA_DIR`.


## Persiapan
Pastikan pustaka `numpy`, `pandas`, `matplotlib`, dan `scipy` tersedia. Kode membaca data dari CSV, memeriksa kolom, lalu membuat grafik dan hasil olahan.

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.interpolate import griddata

DATA_DIR = "."  # ubah ke folder tempat CSV disimpan, jika diperlukan
plt.rcParams["figure.figsize"] = (9, 5)
plt.rcParams["axes.grid"] = True


## Kasus 1 — Pengolahan Data Seismik Refraksi
**Tujuan:** membuat kurva waktu-tempuh terhadap offset, memperkirakan kecepatan dari kemiringan segmen, dan memperlihatkan data first-arrival. CSV memuat offset dan waktu tiba simulasi. Estimasi intercept-time yang sederhana harus ditafsirkan hati-hati karena picking dan identifikasi segmen memengaruhi hasil.

In [ ]:
df_seis = pd.read_csv(os.path.join(DATA_DIR, "data_seismik_refraksi.csv"))
display(df_seis.head())
print(df_seis.describe(include="all"))

fig, ax = plt.subplots()
for shot, g in df_seis.groupby("Shot"):
    ax.scatter(g["Offset_m"], g["First_Arrival_ms"], label=shot)
ax.set(title="Kurva Time–Distance Seismik Refraksi", xlabel="Offset (m)", ylabel="First arrival (ms)")
ax.legend()
plt.show()

# Estimasi kecepatan efektif dari kemiringan garis waktu-jarak untuk tiap shot.
# Karena t(ms) -> s, v = 1 / slope(s/m).
for shot, g in df_seis.groupby("Shot"):
    slope_ms_m, intercept = np.polyfit(g["Offset_m"], g["First_Arrival_ms"], 1)
    v_eff = 1000 / slope_ms_m
    print(f"{shot}: slope={slope_ms_m:.4f} ms/m, kecepatan efektif ~ {v_eff:.1f} m/s")


**Interpretasi awal:** waktu tiba cenderung meningkat terhadap offset. Kemiringan kurva berhubungan terbalik dengan kecepatan rambat. Kurva dengan beberapa segmen linear dapat mengindikasikan gelombang langsung dan gelombang refraksi dari lapisan lebih cepat. Estimasi kecepatan di atas adalah regresi seluruh rentang, bukan pengganti pemilihan segmen first-arrival per lapisan.

## Kasus 2 — Pengolahan Data Geolistrik Resistivitas Schlumberger
**Tujuan:** memplot resistivitas semu terhadap AB/2. CSV berisi AB/2, MN/2, dan resistivitas semu simulasi. Karena data CSV tidak memuat arus dan beda potensial mentah, perhitungan faktor geometri dari pengukuran mentah tidak dilakukan.

In [ ]:
df_geo = pd.read_csv(os.path.join(DATA_DIR, "data_geolistrik_schlumberger.csv"))
display(df_geo.head())

fig, ax = plt.subplots()
ax.semilogx(df_geo["AB2_m"], df_geo["Resistivitas_Semu_ohm_m"], "o-")
ax.set(title="Sounding Resistivitas Semu — Schlumberger", xlabel="AB/2 (m)", ylabel="Resistivitas semu (ohm m)")
plt.show()

df_geo["Perubahan_resistivitas_persen"] = df_geo["Resistivitas_Semu_ohm_m"].pct_change()*100
display(df_geo.head(10))
df_geo.to_csv("hasil_geolistrik.csv", index=False)


**Interpretasi awal:** perubahan resistivitas semu terhadap AB/2 menunjukkan respons gabungan lapisan bawah permukaan pada kedalaman investigasi berbeda. Kurva sounding saja tidak menentukan litologi atau akuifer secara unik; interpretasi kuantitatif ketebalan dan resistivitas lapisan memerlukan inversi 1D serta informasi geologi pendukung.

## Kasus 3 — Pengolahan Data Geomagnetik
**Tujuan:** koreksi variasi harian berdasarkan kolom base station, koreksi IGRF, serta pemisahan regional-residual dan peta kontur. Data adalah grid simulasi.

In [ ]:
df_mag = pd.read_csv(os.path.join(DATA_DIR, "data_geomagnetik.csv"))
display(df_mag.head())

# Koreksi sederhana: kurangi variasi diurnal relatif terhadap nilai referensi awal.
diurnal = df_mag["Base_diurnal_nT"] - df_mag["Base_diurnal_nT"].iloc[0]
df_mag["TMI_setelah_diurnal_nT"] = df_mag["TMI_observed_nT"] - diurnal
df_mag["Anomali_magnetik_nT"] = df_mag["TMI_setelah_diurnal_nT"] - df_mag["IGRF_nT"]

# Regional planar (polinomial orde 1) dan residual
A = np.column_stack([np.ones(len(df_mag)), df_mag["X_m"], df_mag["Y_m"]])
coef, *_ = np.linalg.lstsq(A, df_mag["Anomali_magnetik_nT"], rcond=None)
df_mag["Regional_nT"] = A @ coef
df_mag["Residual_nT"] = df_mag["Anomali_magnetik_nT"] - df_mag["Regional_nT"]

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for ax, col, title in [(axes[0],"Anomali_magnetik_nT","Anomali Magnetik"),(axes[1],"Residual_nT","Anomali Residual")]:
    sc=ax.tricontourf(df_mag["X_m"],df_mag["Y_m"],df_mag[col],levels=15)
    fig.colorbar(sc,ax=ax,label="nT")
    ax.set(title=title,xlabel="X (m)",ylabel="Y (m)")
plt.tight_layout()
plt.show()
df_mag.to_csv("hasil_geomagnetik.csv", index=False)


**Interpretasi awal:** anomali positif/negatif dan gradien tinggi dapat menandakan kontras sifat magnetik atau batas litologi. Hasil koreksi dan pemisahan regional-residual sangat bergantung pada kualitas base station, nilai IGRF yang benar untuk waktu/lokasi, dan metode regional yang dipilih. Data ini simulasi; RTP tidak dilakukan karena parameter magnetisasi dan kondisi data lapangan tidak tersedia.

## Kasus 4 — Pengolahan Data Gravitasi
**Tujuan:** menghitung anomali Bouguer sederhana-lengkap dari kolom gravitasi observasi, normal, elevasi, densitas, dan koreksi medan yang disediakan. Konvensi tanda koreksi harus konsisten dengan definisi data.

In [ ]:
df_grav = pd.read_csv(os.path.join(DATA_DIR, "data_gravitasi.csv"))
display(df_grav.head())

df_grav["FAC_mGal"] = 0.3086 * df_grav["Elevation_m"]
df_grav["Bouguer_correction_mGal"] = 0.04193 * df_grav["Bouguer_density_g_cm3"] * df_grav["Elevation_m"]
df_grav["Anomali_awal_mGal"] = df_grav["Gravity_observed_mGal"] - df_grav["Gravity_normal_mGal"]
df_grav["Anomali_Bouguer_sederhana_mGal"] = df_grav["Anomali_awal_mGal"] + df_grav["FAC_mGal"] - df_grav["Bouguer_correction_mGal"]
df_grav["Anomali_Bouguer_lengkap_mGal"] = df_grav["Anomali_Bouguer_sederhana_mGal"] + df_grav["Terrain_correction_mGal"]

A = np.column_stack([np.ones(len(df_grav)), df_grav["X_m"], df_grav["Y_m"]])
coef, *_ = np.linalg.lstsq(A, df_grav["Anomali_Bouguer_lengkap_mGal"], rcond=None)
df_grav["Regional_mGal"] = A @ coef
df_grav["Residual_mGal"] = df_grav["Anomali_Bouguer_lengkap_mGal"] - df_grav["Regional_mGal"]

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for ax, col, title in [(axes[0],"Anomali_Bouguer_lengkap_mGal","Anomali Bouguer Lengkap"),(axes[1],"Residual_mGal","Residual Gravitasi")]:
    sc=ax.tricontourf(df_grav["X_m"],df_grav["Y_m"],df_grav[col],levels=15)
    fig.colorbar(sc,ax=ax,label="mGal")
    ax.set(title=title,xlabel="X (m)",ylabel="Y (m)")
plt.tight_layout()
plt.show()
df_grav.to_csv("hasil_gravitasi.csv", index=False)


**Interpretasi awal:** anomali gravitasi relatif positif dapat berkaitan dengan kontras densitas positif, sedangkan anomali negatif dapat berkaitan dengan material berdensitas lebih rendah. Ini bukan identifikasi litologi yang unik. Koreksi terrain dalam CSV adalah simulasi; pada survei nyata koreksi membutuhkan topografi/DEM dan prosedur yang sesuai.

## Kasus 5 — Pengolahan Data Magnetotelurik (MT)
**Tujuan:** memvisualisasikan kurva resistivitas semu dan fase terhadap periode untuk tiap stasiun. CSV berisi respons MT simulasi; tidak berisi time series medan mentah atau tensor impedansi kompleks sehingga perhitungan impedansi dan inversi 1D tidak diklaim dari CSV ini.

In [ ]:
df_mt = pd.read_csv(os.path.join(DATA_DIR, "data_magnetotelurik.csv"))
display(df_mt.head())

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for station, g in df_mt.groupby("Station"):
    g=g.sort_values("Period_s")
    axes[0].loglog(g["Period_s"],g["Apparent_Resistivity_ohm_m"],"o-",ms=3,label=station)
    axes[1].semilogx(g["Period_s"],g["Phase_deg"],"o-",ms=3,label=station)
axes[0].set(title="Kurva Resistivitas Semu MT",xlabel="Periode (s)",ylabel="Resistivitas semu (ohm m)")
axes[1].set(title="Kurva Fase MT",xlabel="Periode (s)",ylabel="Fase (derajat)")
axes[0].legend()
axes[1].legend()
plt.tight_layout()
plt.show()

# Ringkasan tiap stasiun
ringkasan_mt=df_mt.groupby("Station").agg(
    rhoa_min=("Apparent_Resistivity_ohm_m","min"),
    rhoa_max=("Apparent_Resistivity_ohm_m","max"),
    phase_mean=("Phase_deg","mean")
).reset_index()
display(ringkasan_mt)
df_mt.to_csv("hasil_ringkasan_mt.csv", index=False)


**Interpretasi awal:** resistivitas semu dan fase berubah terhadap periode karena sensitivitas respons elektromagnetik terhadap struktur bawah permukaan berbeda. Zona konduktif dapat berkaitan dengan lempung, fluida, atau litologi konduktif, tetapi penentuan clay cap/reservoir memerlukan inversi MT 1D/2D/3D dan korelasi geologi. Data ini hanya simulasi.

## Kesimpulan Umum
Notebook ini mendemonstrasikan lima alur pengolahan data geofisika menggunakan CSV simulasi. Grafik dan hasil numerik berguna untuk latihan pemrograman dan memahami alur kerja, tetapi bukan bukti kondisi geologi lokasi nyata. Untuk penelitian atau laporan final, ganti data simulasi dengan data lapangan yang tervalidasi dan dokumentasikan sumber, satuan, koordinat, waktu akuisisi, serta parameter koreksi.